# Flow-DPPO 最小可运行 Demo

本 notebook 对应 `11_flow_DPPO.ipynb`。Flow-DPPO 不再用 PPO 的 ratio clipping 近似信任域，而是利用流模型每步同方差高斯策略的结构，直接计算新旧转移分布的精确 KL，并构造**优势方向感知的不对称 mask**。

参考实现：

- `online_RL/src/online_rl/algorithms/flow_dppo/algorithm.py`；
- `开源模型参考/RL/UniRL/unirl/algorithms/flowdppo.py`。

本例保留 SDE rollout、冻结 old anchor、多轮 replay、importance ratio、解析 KL、mask 和 reference KL；仅将大模型替换为二维小 MLP。


## 1. 三种量不能混为一谈

对已由 old policy 采样的动作 $a_t=x_{t+1}$：

$$r_t=\exp(\log\pi_\theta(a_t|s_t)-\log\pi_{old}(a_t|s_t))$$

仍然用于策略梯度，但它依赖当前采样动作，具有随机性。信任域判断使用同方差高斯之间的解析 KL：

$$D_t^{old,new}=\frac{\|\mu_\theta-\mu_{old}\|_2^2}{2\sigma_t^2}.$$

可选 reference 正则则是另一个量：$D_t^{new,ref}$。old 是本轮数据收集策略，ref 是固定预训练模型，两者不能互换。

高 KL 时只屏蔽继续向外冲的更新：

- $A>0,r>1$：已经增加好动作概率且继续远离 old，屏蔽；
- $A<0,r<1$：已经降低坏动作概率且继续远离 old，屏蔽；
- $A>0,r<1$ 或 $A<0,r>1$：更新方向是在纠正回 old，保留。


In [ ]:
import math
from copy import deepcopy

import torch
import torch.nn as nn

torch.manual_seed(23)
G, T, D = 20, 6, 2
LR = 5e-3
NOISE = 0.35
KL_THRESHOLD = 2e-3
BETA_REF = 1e-3

class TinyFlowPolicy(nn.Module):
    def __init__(self, hidden=48):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(D + 2, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, D),
        )
    def forward(self, x, t, cond):
        return self.net(torch.cat([x, t[:, None], cond], dim=-1))

def reward_fn(x, cond):
    goal = torch.cat([1.2 + cond, -0.8 + 0.25 * cond], dim=-1)
    return -(x - goal).square().sum(-1)

def group_advantages(rewards, eps=1e-6):
    return ((rewards - rewards.mean()) / (rewards.std(correction=0) + eps)).detach()


In [ ]:
@torch.no_grad()
def collect_sde(old, cond):
    """old policy rollout；返回固定 replay 数据。"""
    dt = 1.0 / T
    std_scalar = NOISE * math.sqrt(dt)
    x = torch.randn(cond.shape[0], D)
    states, logps, means, stds = [x.clone()], [], [], []
    for step in range(T):
        t = torch.full((x.shape[0],), step / T)
        mean = x + dt * old(x, t, cond)
        std = torch.full_like(mean, std_scalar)
        dist = torch.distributions.Normal(mean, std)
        action = dist.sample()
        states.append(action.clone())
        logps.append(dist.log_prob(action).sum(-1))
        means.append(mean)
        stds.append(std)
        x = action
    return {
        'states': torch.stack(states, 1),
        'old_logp': torch.stack(logps, 1),
        'old_means': torch.stack(means, 1),
        'stds': torch.stack(stds, 1),
    }

def replay(policy, data, cond):
    """在 old 采样动作上计算 policy 的 logp 与高斯均值。"""
    dt = 1.0 / T
    logps, means = [], []
    for step in range(T):
        x = data['states'][:, step]
        action = data['states'][:, step + 1].detach()
        t = torch.full((x.shape[0],), step / T)
        mean = x + dt * policy(x, t, cond)
        dist = torch.distributions.Normal(mean, data['stds'][:, step])
        logps.append(dist.log_prob(action).sum(-1))
        means.append(mean)
    return torch.stack(logps, 1), torch.stack(means, 1)


In [ ]:
def exact_same_variance_kl(mean_a, mean_b, std):
    """逐样本逐时间步 KL；输入 [G,T,D]，输出 [G,T]。"""
    return ((mean_a - mean_b).square() / (2.0 * std.square().clamp_min(1e-8))).sum(-1)

def flow_dppo_loss(new_logp, old_logp, new_means, old_means, stds, advantages,
                   kl_threshold=KL_THRESHOLD, ref_means=None, beta_ref=BETA_REF):
    adv = advantages[:, None].expand_as(new_logp).detach()
    ratio = (new_logp - old_logp).exp()
    kl_old = exact_same_variance_kl(new_means, old_means, stds)
    high_kl = kl_old >= kl_threshold
    pos_remove = high_kl & (ratio > 1.0) & (adv > 0)
    neg_remove = high_kl & (ratio < 1.0) & (adv < 0)
    remove = pos_remove | neg_remove
    keep = (~remove).detach()

    # torch.where 避免极端 ratio 下出现 inf*0=nan。
    surrogate = -ratio * adv
    policy_loss = torch.where(keep, surrogate, torch.zeros_like(surrogate)).mean()

    ref_kl = new_logp.new_zeros(())
    if ref_means is not None and beta_ref > 0:
        ref_kl = exact_same_variance_kl(new_means, ref_means, stds).mean()
    total = policy_loss + beta_ref * ref_kl
    metrics = {
        'ratio_mean': ratio.detach().mean(),
        'kl_old_mean': kl_old.detach().mean(),
        'high_kl_fraction': high_kl.float().mean(),
        'masked_fraction': remove.float().mean(),
        'pos_removed': pos_remove.float().mean(),
        'neg_removed': neg_remove.float().mean(),
        'ref_kl': ref_kl.detach(),
    }
    return total, metrics, keep, ratio, kl_old


## 2. 先用可控张量验证 mask，而不是直接相信训练曲线

下面构造四个高 KL 元素，依次对应：正优势向外、负优势向外、正优势回归、负优势回归。预期 keep mask 为 `[False, False, True, True]`。


In [ ]:
old_lp_test = torch.zeros(4, 1)
new_lp_test = torch.log(torch.tensor([[1.2], [0.8], [0.8], [1.2]]))
adv_test = torch.tensor([1.0, -1.0, 1.0, -1.0])
old_mean_test = torch.zeros(4, 1, 1)
new_mean_test = torch.ones(4, 1, 1) * 0.2
std_test = torch.ones_like(new_mean_test) * 0.1
_, _, keep_test, ratio_test, kl_test = flow_dppo_loss(
    new_lp_test, old_lp_test, new_mean_test, old_mean_test, std_test, adv_test,
    kl_threshold=0.1, ref_means=None, beta_ref=0.0,
)
print('ratio:', ratio_test.squeeze(1).tolist())
print('KL:', kl_test.squeeze(1).tolist())
print('keep:', keep_test.squeeze(1).tolist())
assert keep_test.squeeze(1).tolist() == [False, False, True, True]


In [ ]:
theta = TinyFlowPolicy()
ref = deepcopy(theta).eval()
for p in ref.parameters():
    p.requires_grad_(False)
optimizer = torch.optim.Adam(theta.parameters(), lr=LR)
cond = torch.zeros(G, 1)

for iteration in range(8):
    # 一次 rollout 建立 old anchor；同一批数据复用多个 update，才能体现 DPPO 信任域。
    old = deepcopy(theta).eval()
    for p in old.parameters():
        p.requires_grad_(False)
    data = collect_sde(old, cond)
    rewards = reward_fn(data['states'][:, -1], cond)
    advantages = group_advantages(rewards)

    for update in range(3):
        new_logp, new_means = replay(theta, data, cond)
        with torch.no_grad():
            _, ref_means = replay(ref, data, cond)
        loss, metrics, keep, ratio, kl_old = flow_dppo_loss(
            new_logp, data['old_logp'], new_means, data['old_means'],
            data['stds'], advantages, ref_means=ref_means,
        )
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(theta.parameters(), 1.0)
        optimizer.step()

    print(f'iter={iteration:02d} reward={rewards.mean().item():+.3f} '
          f'KL_old={metrics["kl_old_mean"].item():.5f} '
          f'masked={metrics["masked_fraction"].item():.2f} '
          f'KL_ref={metrics["ref_kl"].item():.5f}')

assert data['states'].shape == (G, T + 1, D)
assert new_logp.shape == kl_old.shape == keep.shape == (G, T)
assert torch.isfinite(loss)


## 3. 正确实现检查表

- old log-prob 和 old mean 必须在第一次 optimizer step 前冻结，后续多轮 update 不可重算锚点；
- KL 必须使用高斯**转移均值**和相同的转移标准差，不能直接拿速度输出不经步长缩放就比较；
- latent 多维 KL 的严格公式对事件维求和；大型实现若取均值，会改变阈值尺度，阈值必须同步重标定；
- mask 必须 detach，不能让离散判定进入梯度；
- `KL(old,new)` 的 mask 与 `KL(new,ref)` 正则是两个独立机制；
- Flow-DPPO 没有删除 importance ratio，它删除的是统一 ratio clipping。
